In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto.')

for candidate in [
    PROJECT_ROOT / 'code',
    PROJECT_ROOT / 'code' / 'revenue',
    PROJECT_ROOT / 'code' / 'tmdb',
]:
    if candidate.exists() and str(candidate.resolve()) not in sys.path:
        sys.path.append(str(candidate.resolve()))

import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

from experiment_utils import (
    ARTIFACT_DIR,
    MODEL_CONFIGS,
    PREDICTIONS_PATH,
    RESULTS_PATH,
    SUMMARY_PATH,
    TARGET_CONFIGS,
    build_pipeline,
    clean_param_names,
    compute_summary,
    format_summary_display,
    inverse_predictions,
    load_feature_matrix,
    load_processed_movies,
    load_saved_folds,
    make_holdout_split,
    serialize_params,
)

# **Seleção de Modelos para Revenue**

Este notebook concentra a comparação inicial entre os regressores considerados no projeto. Aqui são avaliados `Dummy Regressor`, `Linear Regression`, `KNN Regressor`, `SVR`, `Decision Tree Regressor`, `Random Forest Regressor`, `Gradient Boosting Regressor` e `XGBoost Regressor` diretamente na escala original de `revenue`.

O objetivo é medir, sob um protocolo único, como diferentes famílias de modelos respondem à mesma base e à mesma estratégia de validação. Os resultados por fold, as previsões e os melhores hiperparâmetros são salvos em `data/revenue_model_selection/`, para que os notebooks seguintes consigam aprofundar a análise sem repetir o grid search completo.


In [2]:
df_movies = load_processed_movies()

In [3]:
df_movies

,id_tmdb,title,runtime,adult,belongs_to_collection,budget,Action,Adventure,Animation,Comedy,...,sv,ta,te,th,tn,tr,uk,ur,zh,revenue
0,552524,Lilo & Stitch,108,0,0,100000000,0,0,0,1,...,0,0,0,0,0,0,0,0,0,610800000
1,950387,A Minecraft Movie,101,0,1,150000000,0,1,0,1,...,0,0,0,0,0,0,0,0,0,947000000
2,1257960,सिकंदर,133,0,0,23500000,1,0,0,0,...,0,0,0,0,0,0,0,0,0,24727058
3,574475,Final Destination Bloodlines,110,0,1,50000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,229314062
4,1197306,A Working Man,116,0,0,40000000,1,0,0,0,...,0,0,0,0,0,0,0,0,0,98652557
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6913,12211,Highlander: Endgame,87,0,1,25000000,1,1,0,0,...,0,0,0,0,0,0,0,0,0,15843608
6914,590706,Jiu Jitsu,102,0,0,23000000,1,0,0,0,...,0,0,0,0,0,0,0,0,0,99924
6915,459616,Alad'2,98,0,1,18900000,0,1,0,1,...,0,0,0,0,0,0,0,0,0,19000000
6916,188207,The Legend of Hercules,99,0,0,70000000,1,1,0,0,...,0,0,0,0,0,0,0,0,0,61279452


# **Carregando os Folds**

A divisão externa em 10 folds é reconstruída a partir do arquivo `data/revenue_folds.csv`, garantindo que todos os modelos sejam comparados sobre exatamente as mesmas partições de treino e teste. Isso evita variações artificiais de desempenho causadas por mudanças na amostragem e mantém a comparação metodologicamente justa.


In [4]:
X, y = load_feature_matrix(df_movies)
_, folds_df, _ = load_saved_folds(df_movies)

folds_df

,fold,train_index,test_index
0,0,"[0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 14, 15, 16...","[7, 12, 13, 17, 21, 22, 42, 43, 89, 95, 111, 1..."
1,1,"[0, 1, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 1...","[2, 9, 18, 25, 37, 56, 65, 70, 76, 87, 97, 118..."
2,2,"[0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 1...","[3, 6, 38, 77, 78, 88, 101, 115, 120, 126, 162..."
3,3,"[0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 15...","[8, 14, 19, 20, 24, 35, 39, 53, 67, 68, 86, 94..."
4,4,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[28, 29, 64, 106, 109, 117, 136, 143, 153, 159..."
5,5,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[15, 23, 30, 31, 34, 47, 58, 71, 72, 93, 102, ..."
6,6,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[59, 80, 91, 92, 107, 116, 138, 164, 165, 176,..."
7,7,"[0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 26, 32, 36, 44, 45, 46, 48, 49, 50, 52, 61..."
8,8,"[0, 1, 2, 3, 4, 6, 7, 8, 9, 12, 13, 14, 15, 17...","[5, 10, 11, 16, 27, 33, 40, 51, 55, 57, 60, 62..."
9,9,"[1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...","[0, 4, 41, 54, 73, 74, 75, 85, 96, 100, 105, 1..."


# **Grid Search com os Regressores Considerados**

Cada modelo é avaliado nos 10 folds externos, sempre com um holdout interno fixo para a escolha de hiperparâmetros via `GridSearchCV`. Essa separação entre validação interna e teste externo ajuda a reduzir vazamento de informação e deixa mais clara a diferença entre seleção de configuração e avaliação final.

Ao final, o notebook salva três artefatos complementares: métricas por fold, predições por fold e um resumo agregado por versão do alvo e modelo. Esse conjunto é suficiente para sustentar tanto a escolha do melhor regressor quanto as análises interpretativas realizadas depois.


In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

fold_results = []
prediction_records = []

for target_name, target_config in TARGET_CONFIGS.items():
    print(f'=== Versão do alvo: {target_name} ===')

    for model_name, config in MODEL_CONFIGS.items():
        print(f'--- {model_name} ---')

        for idx, row in folds_df.iterrows():
            train_idx = row['train_index']
            test_idx = row['test_index']

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            y_train_model = target_config['forward'](y_train)

            holdout_cv = make_holdout_split(X_train)

            grid_search = GridSearchCV(
                estimator=build_pipeline(clone(config['estimator'])),
                param_grid=config['param_grid'],
                scoring='neg_mean_squared_error',
                cv=holdout_cv,
                refit=True,
                verbose=0,
            )

            grid_search.fit(X_train, y_train_model)

            best_model = grid_search.best_estimator_
            best_params = clean_param_names(grid_search.best_params_)
            y_pred_model = best_model.predict(X_test)
            y_pred = inverse_predictions(y_pred_model, target_name)

            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            fold_results.append({
                'target_version': target_name,
                'model': model_name,
                'fold': idx,
                'mse': mse,
                'rmse': rmse,
                'mae': mae,
                'r2': r2,
                'best_params_json': serialize_params(best_params),
            })

            fold_predictions = pd.DataFrame({
                'row_index': test_idx,
                'id_tmdb': df_movies.iloc[test_idx]['id_tmdb'].values,
                'title': df_movies.iloc[test_idx]['title'].values,
                'target_version': target_name,
                'model': model_name,
                'fold': idx,
                'y_true': y_test,
                'y_pred': y_pred,
            })
            fold_predictions['residual'] = fold_predictions['y_true'] - fold_predictions['y_pred']
            fold_predictions['abs_error'] = fold_predictions['residual'].abs()
            prediction_records.append(fold_predictions)

            print(f'Fold {idx}: RMSE={rmse:.6f} | MAE={mae:.6f} | R²={r2:.6f}')
            print(f' - Best Params: {best_params}\n')

results_df = pd.DataFrame(fold_results)
predictions_df = pd.concat(prediction_records, ignore_index=True)
summary_table_df = compute_summary(results_df)
summary_df = summary_table_df.reset_index()
summary_display_df = format_summary_display(summary_table_df)

results_df.to_csv(RESULTS_PATH, index=False)
predictions_df.to_csv(PREDICTIONS_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)

print(f'Resultados salvos em: {RESULTS_PATH}')
print(f'Predições salvas em: {PREDICTIONS_PATH}')
print(f'Resumo salvo em: {SUMMARY_PATH}')

summary_display_df

In [ ]:
summary_df.sort_values(['target_version', 'mean_rmse'])